# Práctica 6 Web Scraping

Alumno: Salvador Calderón Martínez

En esta versión trabajaremos con la página de Wikipedia "List of best-selling video games", que contiene una tabla con los videojuegos más vendidos de la historia: título, ventas, serie, plataformas, año de lanzamiento, desarrollador y editor. Se incluye el archivo .csv del repositorio de github.


In [ ]:
# Requests sirve para enviar solicitudes HTTP a servidores web, permite interactuar con páginas web.
import requests

# BeautifulSoup sirve para analizar y extraer datos de documentos HTML y XML.
from bs4 import BeautifulSoup

import pandas as pd

In [ ]:
# 1. Hacer la petición a la página objetivo
url = "https://en.wikipedia.org/wiki/List_of_best-selling_video_games"

# Wikipedia requiere un User-Agent identificable; sin este encabezado puede rechazar la petición.
headers = {"User-Agent": "Mozilla/5.0 (compatible; CursoScraping/1.0)"}

#Envía una petición GET al servidor. El servidor entrega todo el HTML de la página.
response = requests.get(url, headers=headers)

# Garantiza que caracteres especiales se interpreten sin errores.
response.encoding = 'utf-8'

In [ ]:
# 2. Convertir el texto HTML a un objeto parseable con BeautifulSoup
# Toma el texto HTML de la respuesta y lo analiza con el motor "html.parser".
# Un objeto parseable es un dato en formato de texto plano que tiene la estructura correcta para ser analizado
texto = BeautifulSoup(response.text, "html.parser")
texto

In [ ]:
# 3. Localizar la tabla de videojuegos más vendidos (etiqueta <table class="wikitable">)
# Busca en todo el documento HTML la primera tabla que cumpla la condición.
# find() (a diferencia de find_all) devuelve solo la primera coincidencia, que es la tabla que nos interesa.

tabla = texto.find("table", class_="wikitable")

# find_all("tr") localiza todas las filas de la tabla; con [1:] descartamos la fila de encabezado.
filas = tabla.find_all("tr")[1:]
filas

In [ ]:
datos = []

# 4. Iterar sobre cada fila para "pescar" los datos sueltos
for fila in filas:
    celdas = fila.find_all(["td", "th"])

    # Algunas filas no incluyen la columna "Rank" porque está fusionada (rowspan) con la fila anterior
    # (ocurre cuando dos juegos empatan en el mismo puesto). Por eso tomamos siempre las últimas 8 celdas,
    # que son las que sí están garantizadas en cada fila: Título, Ventas, Serie, Plataforma(s),
    # Año, Desarrollador(es), Editor(es) y Referencia.
    celdas = celdas[-8:]

    titulo = celdas[0].get_text(strip=True)
    ventas_millones = celdas[1].get_text(strip=True)
    serie = celdas[2].get_text(strip=True)
    plataformas = celdas[3].get_text(strip=True)
    anio_lanzamiento = celdas[4].get_text(strip=True)
    desarrollador = celdas[5].get_text(strip=True)
    editor = celdas[6].get_text(strip=True)

    datos.append({
        "titulo": titulo,
        "ventas_millones": ventas_millones,
        "serie": serie,
        "plataformas": plataformas,
        "anio_lanzamiento": anio_lanzamiento,
        "desarrollador": desarrollador,
        "editor": editor
    })

In [ ]:
# 5. Convertir la lista de diccionarios a un DataFrame (Estructura Tidy Data)
df = pd.DataFrame(datos)
df

In [ ]:
%%writefile scraper.py
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://en.wikipedia.org/wiki/List_of_best-selling_video_games"
headers = {"User-Agent": "Mozilla/5.0 (compatible; CursoScraping/1.0)"}
response = requests.get(url, headers=headers)
response.encoding = 'utf-8'

soup = BeautifulSoup(response.text, "html.parser")
tabla = soup.find("table", class_="wikitable")
filas = tabla.find_all("tr")[1:]

datos = []
for fila in filas:
    celdas = fila.find_all(["td", "th"])[-8:]

    datos.append({
        "titulo": celdas[0].get_text(strip=True),
        "ventas_millones": celdas[1].get_text(strip=True),
        "serie": celdas[2].get_text(strip=True),
        "plataformas": celdas[3].get_text(strip=True),
        "anio_lanzamiento": celdas[4].get_text(strip=True),
        "desarrollador": celdas[5].get_text(strip=True),
        "editor": celdas[6].get_text(strip=True)
    })

df = pd.DataFrame(datos)
df.to_csv("videojuegos_mas_vendidos.csv", index=False)
print("Archivo videojuegos_mas_vendidos.csv creado.")

In [ ]:
import pandas as pd

In [ ]:
# URL RAW GitHub Salvador
url_raw ="https://raw.githubusercontent.com/SalvadorCM786/Web-Scraping/refs/heads/main/videojuegos_mas_vendidos.csv"

In [ ]:
# Cada vez que ejecutes este script, se descargará el archivo CSV desde la URL RAW de GitHub y se guardará en tu directorio de trabajo actual. 
df = pd.read_csv(url_raw)
df